# Clusterize Successful Scala

Rewrite a successful sample-data Scala job into a cluster-ready Scala file, preserve the existing Scala output handling by default, and generate the commands to upload, compile, package, and submit it on the cluster accessed with `ssh lab`.


In [1]:
from pathlib import Path
import json
import sys

NOTEBOOK_ROOT = Path('/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook')
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from rdpro_section_codegen.clusterize_scala import clusterize_scala_text
from rdpro_section_codegen.file_utils import read_text, write_text


/Users/clockorangezoe/miniconda3/envs/geo_llm_spark/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Local input/output for the rewritten Scala
input_scala = Path('/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/one_shot_output_sectional_11.scala')
output_scala = Path('/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/cluster_ready_one_shot_output_sectional_11.scala')

# Keep the raster fixed and vary only the vector granularity.
fixed_raster_path = 'hdfs:///user/zshan011/geoai/nlcd/Annual_NLCD_LndCov_2024_CU_C1V1.tif'
dataset_profiles = {
    'boston': {
        'raster_paths': [fixed_raster_path],
        'vector_paths': ['hdfs:///user/zshan011/geoai/boston_neighborhood_boundaries/Boston_Neighborhood_Boundaries_sample.shp'],
        'zone_field': 'name',
    },
    'states': {
        'raster_paths': [fixed_raster_path],
        'vector_paths': ['hdfs:///user/zshan011/geoai/states/states.shp'],
        'zone_field': 'NAME',
    },
    'counties': {
        'raster_paths': [fixed_raster_path],
        'vector_paths': ['hdfs:///user/zshan011/geoai/us_counties/us_counties.shp'],
        'zone_field': 'NAME',
    },
}
active_dataset = 'boston'
selected_dataset = dataset_profiles[active_dataset]
raster_paths = selected_dataset['raster_paths']
vector_paths = selected_dataset['vector_paths']
zone_field = selected_dataset['zone_field']

# Leave as None to preserve the output target already encoded in the Scala job.
cluster_output_path_override = None

# Cluster execution settings
cluster_master = 'spark://ec-hn.cs.ucr.edu:7077'
remote_host_alias = 'lab'
remote_workdir = '~/rdpro_codegen_demo'
remote_scala_name = 'cluster_ready_one_shot_output_sectional_11.scala'
remote_classes_dir = 'classes_cluster_ready'
remote_jar_name = 'cluster_ready_one_shot_output_sectional_11.jar'
main_class = 'GeoJobMain'

# If the cluster already has all jars available, keep these empty.
cluster_classpath_entries = []
spark_submit_extra_args = []

keep_step_markers = False
print('active_dataset =', active_dataset)
print('zone_field =', zone_field)


In [3]:
scala_text = read_text(input_scala)
cluster_scala, summary = clusterize_scala_text(
    scala_text=scala_text,
    raster_paths=raster_paths,
    vector_paths=vector_paths,
    output_path=cluster_output_path_override,
    master=cluster_master,
    keep_step_markers=keep_step_markers,
)
cluster_scala = cluster_scala.replace('getAs[String]("name")', f'getAs[String]("{zone_field}")')
write_text(output_scala, cluster_scala)
summary = {
    'input_scala': str(input_scala),
    'output_scala': str(output_scala),
    'active_dataset': active_dataset,
    'zone_field': zone_field,
    **summary,
}
print(json.dumps(summary, indent=2))


{
  "input_scala": "/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/one_shot_output_sectional.scala",
  "output_scala": "/Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/cluster_ready_output.scala",
  "raster_paths": [
    "/path/to/cluster/input_raster.tif"
  ],
  "vector_paths": [
    "/path/to/cluster/input_vector.shp"
  ],
  "output_path": "/path/to/cluster/output/result.csv",
  "master": "spark://ec-hn.cs.ucr.edu:7077",
  "removed_actions": {
    "LOAD_DATA": [
      "val firstRaster = raster.take(1).headOption.getOrElse(",
      "if (vector.take(1).isEmpty) {",
      "println(\"__STEP__ LOAD_DATA_done\")"
    ],
    "TYPE_CHECK": [
      "case IntegerType => println(\"Pixel type is IntegerType as expected.\")",
      "case FloatType => println(\"Pixel type is FloatType, expected IntegerType.\")",
      "case DoubleType => println(\"Pixel type is DoubleType, expected IntegerType.\")",


In [4]:
# Preview the rewritten Scala
print('\n'.join(cluster_scala.splitlines()[:160]))


import org.apache.spark.SparkContext
import org.apache.spark.sql.SparkSession
import org.apache.spark.SparkConf
import org.apache.spark.sql.types.{ArrayType, FloatType, IntegerType, DoubleType}
import org.apache.spark.rdd.{CoGroupedRDD, RDD}
import edu.ucr.cs.bdlab.beast._
import edu.ucr.cs.bdlab.raptor.RaptorMixin._
import edu.ucr.cs.bdlab.raptor.RaptorMixin.RasterReadMixinFunctions
import edu.ucr.cs.bdlab.raptor.RasterVectorOperations
import edu.ucr.cs.bdlab.beast.cg.SpatialDataTypes.RasterRDD
import edu.ucr.cs.bdlab.beast.io.tiff.TiffConstants
import edu.ucr.cs.bdlab.beast.common.BeastOptions
import edu.ucr.cs.bdlab.beast.geolite.{ITile, Feature, IFeature, RasterFeature, RasterMetadata}
import edu.ucr.cs.bdlab.raptor.{GeoTiffWriter, RasterOperationsFocal, RasterOperationsLocal, RasterOperationsGlobal, RaptorJoin,RaptorJoinFeature }
import java.net.URI
import java.nio.file.{Paths, Files}
import org.apache.hadoop.conf.Configuration
import org.apache.hadoop.fs.{FileSystem, Path => HPat

In [5]:
classpath_arg = ':'.join(cluster_classpath_entries)
remote_scala_path = f"{remote_workdir}/{remote_scala_name}"
remote_jar_path = f"{remote_workdir}/{remote_jar_name}"

scp_cmd = f"scp {output_scala} {remote_host_alias}:{remote_scala_path}"

compile_cmd = (
    f"cd {remote_workdir} && mkdir -p {remote_classes_dir} && "
    + (
        f'scalac -classpath "{classpath_arg}" -d {remote_classes_dir} {remote_scala_name}'
        if classpath_arg
        else f'scalac -d {remote_classes_dir} {remote_scala_name}'
    )
)

jar_cmd = f"cd {remote_workdir} && jar cf {remote_jar_name} -C {remote_classes_dir} ."

spark_extra = ' '.join(spark_submit_extra_args).strip()
spark_submit_cmd = (
    f"cd {remote_workdir} && spark-submit --master {cluster_master} "
    + (spark_extra + ' ' if spark_extra else '')
    + f"--class {main_class} {remote_jar_name}"
)

print('scp command:')
print(scp_cmd)
print('\nssh to cluster:')
print(f'ssh {remote_host_alias}')
print('\ncompile on cluster:')
print(compile_cmd)
print('\npackage jar on cluster:')
print(jar_cmd)
print('\nsubmit on cluster:')
print(spark_submit_cmd)


scp command:
scp /Users/clockorangezoe/Documents/phd_projects/code/geoAI/LangGraph/Notebook/rdpro_section_codegen/cluster_ready_output.scala lab:~/rdpro_codegen_demo/cluster_ready_output.scala

ssh to cluster:
ssh lab

compile on cluster:
cd ~/rdpro_codegen_demo && mkdir -p classes_cluster_ready && scalac -d classes_cluster_ready cluster_ready_output.scala

package jar on cluster:
cd ~/rdpro_codegen_demo && jar cf cluster_ready_output.jar -C classes_cluster_ready .

submit on cluster:
cd ~/rdpro_codegen_demo && spark-submit --master spark://ec-hn.cs.ucr.edu:7077 --class GeoJobMain cluster_ready_output.jar


## Suggested Cluster Sequence

1. Run the rewrite cells.
2. Copy the Scala file with the printed `scp` command.
3. Log in with `ssh lab`.
4. Run the printed cluster-side `scalac` command.
5. Run the printed `jar` command.
6. Run the printed `spark-submit` command.
